In [20]:
# %% Imports
import pandas as pd
import numpy as np
import logging
import os
import warnings
import seaborn as sns
import sys
from pathlib import Path
# Suppress MLflow logs BEFORE importing mlflow
os.environ["MLFLOW_TRACKING_SILENCE_DEPRECATION_WARNINGS"] = "true"
import mlflow
import numpy as np
import torch
from mlflow.tracking import MlflowClient
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("alembic").setLevel(logging.ERROR)
logging.getLogger("mlflow.store.db.utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*filesystem tracking backend.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="mlflow")
# Add project root to path


In [21]:
def print_model_comparison_table() -> pd.DataFrame:
    """Print a formatted comparison table for models in MODELS_CONFIG."""
    client = get_mlflow_client()

    # Collect data for each model
    rows = []
    for key, cfg in MODELS_CONFIG.items():
        data = {}
        run = client.get_run(cfg["run_id"])
        metrics = run.data.metrics
        params = run.data.params
        data["Model"] = cfg["run_name"]
        data.update(params)
        data.update(metrics)
        rows.append(data)
    # Print table header
    return pd.DataFrame(rows)

In [27]:
def get_mlflow_client() -> MlflowClient:
    """Get configured MLflow client."""
    mlflow.set_tracking_uri(MLFLOW_URI)
    return MlflowClient()

PROJECT_ROOT = Path('/home/davidrfb/Documents/Mano/')

sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

# %% Configuration
# MLflow paths
MLFLOW_URI = f"sqlite:///{PROJECT_ROOT / 'models/mlflow.db'}"
MLFLOW_ARTIFACTS = f"file://{PROJECT_ROOT / 'models/mlartifacts'}"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Models to compare
MODELS_CONFIG = {
    "landmark_mlp": {
        "run_name": "inquisitive-robin-2",
        "run_id": "c6d5eb3a9c544c04a8f6cefca6dc3e72",
        "experiment": "Landmark_only",
        "description": "MLP on hand landmarks (56 features)",
        "model_type": "landmark",  # Uses landmark preprocessing
    },
    "mobilenet_v2": {
        "run_name": "mobilenetv2_png_landmark",
        "run_id": "21e24b44d3f6477c89c6acf2d9c11ff4",
        "experiment": "CNN_v2",
        "description": "MobileNetV2 on raw images (224x224)",
        "model_type": "image",  # Uses image preprocessing
    },
}
DATA_DIR_LANDMARKS = PROJECT_ROOT / "data/raw_landmarks"
DATA_DIR_PHOTOS = PROJECT_ROOT / "data/raw_photo_landmark"



/home/davidrfb/Documents/Mano


In [28]:
client = get_mlflow_client()

In [30]:
print_model_comparison_table().T

,0,1
Model,inquisitive-robin-2,mobilenetv2_png_landmark
model,static,mobilenet_v2
data_dir,data/raw_landmarks,data/raw_photo_landmark
letters,all,all
features,xy_angles,images (N/A)
epochs,100,100
lr,0.01,0.001
batch_size,32,32
hidden_dim,128,128
patience,20,5
